In [20]:
from datasets import load_dataset
import tqdm as notebook_tqdm
from collections import Counter
import torch
import torch.nn as nn
from torch.nn import functional as F
import tqdm
from tqdm import tqdm
from progress_table import ProgressTable


In [29]:
dataset = load_dataset("Trelis/tiny-shakespeare")
train_data = dataset["train"]['Text']
test_data = dataset["test"]['Text']
char_count = sum(len(line) for line in train_data)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('train length:', char_count)
print(device)

train length: 1222354
cuda


In [3]:
chars = sorted(list(set("".join(train_data))))
vocab_size = len(chars)
print(vocab_size)
print("".join(chars))


65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [4]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hello"))
print(decode(encode("hello")))

[46, 43, 50, 50, 53]
hello


In [5]:
train = torch.tensor(encode("".join(train_data)), dtype=torch.long)
test = torch.tensor(encode("".join(test_data)), dtype=torch.long)
print(train.shape, train.dtype)
print(train[:100])
print(test.shape, test.dtype)
print(test[:100])


torch.Size([1222354]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])
torch.Size([119020]) torch.int64
tensor([32, 30, 13, 26, 21, 27, 10,  0, 21, 57,  1, 58, 46, 47, 57,  1, 63, 53,
        59, 56,  1, 57, 54, 43, 43, 42, 47, 52, 45, 12,  1, 52, 39, 63,  6,  1,
        58, 46, 43, 52,  6,  1, 45, 53, 53, 42,  1, 52, 47, 45, 46, 58,  1, 53,
        59, 56,  1, 54, 39, 56, 58,  2,  0,  0, 28, 17, 32, 30, 33, 15, 20, 21,
        27, 10,  0, 14, 43,  1, 54, 39, 58, 47, 43, 52, 58,  6,  1, 45, 43, 52,
        58, 50, 43, 51, 43, 52, 11,  1, 21,  1])


In [31]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train if split == 'train' else test
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # in
    x = torch.stack([data[i:i+block_size] for i in ix]).to(device)
    # target
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]).to(device)
    return x, y

xb, yb = get_batch('train')
print(xb.shape, yb.shape)
print(xb)
print(yb)

torch.Size([4, 8]) torch.Size([4, 8])
tensor([[39, 57,  6,  1, 58, 46, 39, 58],
        [58, 46, 43,  1, 57, 47, 52,  0],
        [53, 63, 57,  1, 61, 47, 58, 46],
        [53, 56,  1, 21,  1, 49, 52, 53]], device='cuda:0')
tensor([[57,  6,  1, 58, 46, 39, 58,  1],
        [46, 43,  1, 57, 47, 52,  0, 27],
        [63, 57,  1, 61, 47, 58, 46,  1],
        [56,  1, 21,  1, 49, 52, 53, 61]], device='cuda:0')


In [40]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        
        logits = self.token_embedding_table(idx).to(device)
        if targets is None:
            loss = None
            return logits
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=-1).to(device)
        return idx

m = BigramLanguageModel(vocab_size)
m.to(device)
logits, loss = m(xb, yb)
print(logits.shape)
print(logits)


torch.Size([256, 65])
tensor([[-0.3560,  1.5867, -0.2606,  ..., -0.4086, -1.2869,  1.1061],
        [-0.6377, -0.5192,  0.5863,  ..., -0.4548,  0.3397, -0.7428],
        [ 0.6229,  0.2997,  0.3521,  ..., -0.2161,  0.1094,  0.6362],
        ...,
        [ 0.5115,  0.1767, -1.8459,  ..., -0.8176,  0.2011,  0.0419],
        [-0.7231,  1.4454, -0.9194,  ..., -1.2476, -2.3829,  0.7340],
        [ 0.6229,  0.2997,  0.3521,  ..., -0.2161,  0.1094,  0.6362]],
       device='cuda:0', grad_fn=<ViewBackward0>)


In [41]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=100)[0].tolist()))


;xbDcRZyNwcDwf,ZT,OLob,yHsK
j:!Pjo'bBFXB?3eXaSKgO-3Z&M:c?gLTauhX:YVUJthhfNuyq&PCEv.tbaF-:XlNDcaLeWaw


In [42]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
batch_size = 32
for i in tqdm(range(10000), desc="Training"):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print("Final loss:", loss.item())

Training: 100%|██████████| 10000/10000 [00:04<00:00, 2366.04it/s]

Final loss: 2.5060629844665527


In [45]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=100)[0].tolist()))



W-ndo whth eiibyo the m dourive we higend t so mower; te

AN ad nterupt f s ar iris! m:

Thiny aler
